In [9]:
import pandas as pd
pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 1000)
newDf = pd.read_parquet("./NQ_1min_output/continuous/NQ_continuous_raw.parquet")
newDf = newDf[newDf["datetime"] <= "2020-10-18 23:00:00"]
newDf = newDf[newDf["datetime"] >= "2020-10-16 20:58:00"]
# newDf = newDf[newDf["open"] == 11816.75]
# newDf = newDf[newDf["high"] == 11850.25]
newDf


,datetime,contract,open,high,low,close,volume
4489733,2020-10-16 20:58:00,NQ 12-20,11795.00,11798.50,11794.75,11797.25,85
4489734,2020-10-16 20:59:00,NQ 12-20,11797.00,11797.50,11793.75,11795.25,206
4489735,2020-10-17 18:01:00,NQ 12-20,11795.25,11795.25,11795.25,11795.25,2
4489736,2020-10-17 18:07:00,NQ 12-20,11795.25,11795.25,11795.25,11795.25,1
4489737,2020-10-17 20:48:00,NQ 12-20,11795.25,11795.25,11795.25,11795.25,1
4489738,2020-10-17 20:51:00,NQ 12-20,11795.25,11795.25,11795.25,11795.25,1
4489739,2020-10-17 23:23:00,NQ 12-20,11795.25,11795.25,11795.25,11795.25,1
4489740,2020-10-18 09:01:00,NQ 12-20,11795.25,11795.25,11795.25,11795.25,1
4489741,2020-10-18 18:52:00,NQ 12-20,11795.25,11921.00,11795.25,11894.75,36243
4489742,2020-10-18 18:53:00,NQ 12-20,11894.50,11897.50,11890.50,11895.00,86


In [2]:
import pandas as pd
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)
oldDf = pd.read_parquet("./nq_1min.parquet")
oldDf = oldDf[oldDf.index <= "2020-10-18 23:00:00"]
oldDf = oldDf[oldDf.index >= "2020-10-16 20:58:00"]
oldDf

,open,high,low,close,volume
datetime,,,,,
2020-10-16 20:58:00,11795.00,11798.50,11794.75,11797.25,85
2020-10-16 20:59:00,11797.00,11797.50,11793.75,11795.25,206
2020-10-18 22:00:00,11816.75,11850.25,11816.75,11835.25,759
2020-10-18 22:01:00,11836.75,11840.50,11831.50,11831.75,255
2020-10-18 22:02:00,11832.75,11839.75,11830.25,11839.25,179
2020-10-18 22:03:00,11838.50,11851.25,11836.50,11849.50,381
2020-10-18 22:04:00,11849.00,11850.25,11845.75,11847.00,272
2020-10-18 22:05:00,11847.25,11858.50,11846.00,11853.25,487
2020-10-18 22:06:00,11853.75,11862.00,11853.25,11860.00,318


In [10]:
import pandas as pd
import numpy as np


# ============================================================
# FILES
# ============================================================

NEW_FILE = "./NQ_1min_output/continuous/NQ_continuous_raw.parquet"
OLD_FILE = "./nq_1min.parquet"


# ============================================================
# LOAD
# ============================================================

print("=" * 70)
print("LOADING DATA")
print("=" * 70)

newDf = pd.read_parquet(NEW_FILE)
oldDf = pd.read_parquet(OLD_FILE)

# Old dataset has datetime as index
oldDf = oldDf.reset_index()

# Standardize datetime
newDf["datetime"] = pd.to_datetime(newDf["datetime"])
oldDf["datetime"] = pd.to_datetime(oldDf["datetime"])

# Sort
newDf = newDf.sort_values("datetime").reset_index(drop=True)
oldDf = oldDf.sort_values("datetime").reset_index(drop=True)


# ============================================================
# BASIC INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("BASIC DATASET INFORMATION")
print("=" * 70)

print("\nNEW DATASET")
print("Rows:", len(newDf))
print("First:", newDf["datetime"].min())
print("Last: ", newDf["datetime"].max())

print("\nOLD DATASET")
print("Rows:", len(oldDf))
print("First:", oldDf["datetime"].min())
print("Last: ", oldDf["datetime"].max())


# ============================================================
# DUPLICATES
# ============================================================

print("\n" + "=" * 70)
print("DUPLICATE TIMESTAMPS")
print("=" * 70)

new_dup_count = newDf["datetime"].duplicated().sum()
old_dup_count = oldDf["datetime"].duplicated().sum()

print("NEW duplicate rows:", new_dup_count)
print("OLD duplicate rows:", old_dup_count)

if new_dup_count > 0:
    print("\nNEW duplicates:")
    print(
        newDf[
            newDf["datetime"].duplicated(keep=False)
        ].head(20)
    )

if old_dup_count > 0:
    print("\nOLD duplicates:")
    print(
        oldDf[
            oldDf["datetime"].duplicated(keep=False)
        ].head(20)
    )


# ============================================================
# TIMESTAMP SET COMPARISON
# ============================================================

print("\n" + "=" * 70)
print("TIMESTAMP COMPARISON")
print("=" * 70)

new_times = set(newDf["datetime"])
old_times = set(oldDf["datetime"])

missing_from_old = sorted(new_times - old_times)
missing_from_new = sorted(old_times - new_times)

common_times = new_times & old_times

print("Timestamps in NEW:", len(new_times))
print("Timestamps in OLD:", len(old_times))
print("Common timestamps:", len(common_times))

print(
    "Present in NEW but missing from OLD:",
    len(missing_from_old)
)

print(
    "Present in OLD but missing from NEW:",
    len(missing_from_new)
)


# ============================================================
# GROUP MISSING TIMESTAMPS
# ============================================================

def group_consecutive_timestamps(timestamps):

    if len(timestamps) == 0:
        return pd.DataFrame(
            columns=[
                "start",
                "end",
                "minutes"
            ]
        )

    s = (
        pd.Series(pd.to_datetime(timestamps))
        .sort_values()
        .reset_index(drop=True)
    )

    groups = (
        s.diff() != pd.Timedelta(minutes=1)
    ).cumsum()

    result = (
        s.groupby(groups)
        .agg(
            start="min",
            end="max"
        )
        .reset_index(drop=True)
    )

    result["minutes"] = (
        (
            result["end"] - result["start"]
        )
        / pd.Timedelta(minutes=1)
    ).astype(int) + 1

    return result


missing_old_ranges = group_consecutive_timestamps(
    missing_from_old
)

missing_new_ranges = group_consecutive_timestamps(
    missing_from_new
)

print("\nMissing from OLD - first 30 ranges:")
print(missing_old_ranges.head(30))

print("\nMissing from NEW - first 30 ranges:")
print(missing_new_ranges.head(30))


# ============================================================
# MERGE COMMON TIMESTAMPS
# ============================================================

print("\n" + "=" * 70)
print("OHLCV COMPARISON")
print("=" * 70)

columns_new = [
    "datetime",
    "contract",
    "open",
    "high",
    "low",
    "close",
    "volume"
]

columns_old = [
    "datetime",
    "open",
    "high",
    "low",
    "close",
    "volume"
]

merged = pd.merge(
    newDf[columns_new],
    oldDf[columns_old],
    on="datetime",
    how="inner",
    suffixes=("_new", "_old")
)

print("Common rows:", len(merged))


# ============================================================
# EXACT MATCHES
# ============================================================

for col in ["open", "high", "low", "close", "volume"]:

    merged[f"{col}_match"] = (
        merged[f"{col}_new"]
        ==
        merged[f"{col}_old"]
    )


merged["all_match"] = (
    merged["open_match"]
    &
    merged["high_match"]
    &
    merged["low_match"]
    &
    merged["close_match"]
    &
    merged["volume_match"]
)


# ============================================================
# MATCH SUMMARY
# ============================================================

perfect_matches = merged["all_match"].sum()
any_mismatch = (~merged["all_match"]).sum()

print("\nPerfect OHLCV matches:", perfect_matches)
print("Rows with any mismatch:", any_mismatch)

print("\nColumn mismatches:")

for col in ["open", "high", "low", "close", "volume"]:

    count = (
        ~merged[f"{col}_match"]
    ).sum()

    print(
        f"{col:>8}: {count}"
    )


# ============================================================
# DIFFERENCES
# ============================================================

for col in ["open", "high", "low", "close", "volume"]:

    merged[f"{col}_diff"] = (
        merged[f"{col}_new"]
        -
        merged[f"{col}_old"]
    )


# ============================================================
# PRICE DIFFERENCES IN TICKS
# ============================================================

for col in ["open", "high", "low", "close"]:

    merged[f"{col}_ticks"] = (
        merged[f"{col}_diff"].abs()
        /
        0.25
    )


merged["max_price_ticks"] = (
    merged[
        [
            "open_ticks",
            "high_ticks",
            "low_ticks",
            "close_ticks"
        ]
    ]
    .max(axis=1)
)


# ============================================================
# VOLUME DIFFERENCES
# ============================================================

merged["volume_diff_abs"] = (
    merged["volume_diff"].abs()
)

merged["volume_pct_diff"] = (
    merged["volume_diff_abs"]
    /
    merged["volume_old"].replace(0, np.nan)
    *
    100
)


# ============================================================
# MISMATCH DATASET
# ============================================================

mismatches = merged[
    ~merged["all_match"]
].copy()


print("\n" + "=" * 70)
print("MISMATCH SEVERITY")
print("=" * 70)

price_mismatches = merged[
    merged["max_price_ticks"] > 0
]

print(
    "Rows with price mismatch:",
    len(price_mismatches)
)

print(
    "Rows with <= 1 tick difference:",
    (
        (price_mismatches["max_price_ticks"] <= 1)
    ).sum()
)

print(
    "Rows with 2-4 tick difference:",
    (
        (price_mismatches["max_price_ticks"] > 1)
        &
        (price_mismatches["max_price_ticks"] <= 4)
    ).sum()
)

print(
    "Rows with > 4 tick difference:",
    (
        price_mismatches["max_price_ticks"] > 4
    ).sum()
)


# ============================================================
# WORST PRICE MISMATCHES
# ============================================================

print("\n" + "=" * 70)
print("WORST PRICE MISMATCHES")
print("=" * 70)

worst_price = merged.sort_values(
    "max_price_ticks",
    ascending=False
)

print(
    worst_price[
        [
            "datetime",
            "contract",
            "open_new",
            "open_old",
            "high_new",
            "high_old",
            "low_new",
            "low_old",
            "close_new",
            "close_old",
            "max_price_ticks"
        ]
    ].head(50)
)


# ============================================================
# WORST VOLUME MISMATCHES
# ============================================================

print("\n" + "=" * 70)
print("WORST VOLUME MISMATCHES")
print("=" * 70)

worst_volume = merged.sort_values(
    "volume_diff_abs",
    ascending=False
)

print(
    worst_volume[
        [
            "datetime",
            "contract",
            "volume_new",
            "volume_old",
            "volume_diff",
            "volume_pct_diff"
        ]
    ].head(50)
)


# ============================================================
# MISMATCH CLUSTERS
# ============================================================

mismatch_times = merged.loc[
    ~merged["all_match"],
    "datetime"
].tolist()

mismatch_ranges = group_consecutive_timestamps(
    mismatch_times
)

print("\n" + "=" * 70)
print("MISMATCH CLUSTERS")
print("=" * 70)

print(
    mismatch_ranges
    .sort_values(
        "minutes",
        ascending=False
    )
    .head(50)
)


# ============================================================
# MISMATCHES BY CONTRACT
# ============================================================

print("\n" + "=" * 70)
print("MISMATCHES BY CONTRACT")
print("=" * 70)

print(
    mismatches[
        "contract"
    ]
    .value_counts()
    .head(50)
)


# ============================================================
# SAVE REPORT FILES
# ============================================================

print("\n" + "=" * 70)
print("SAVING REPORTS")
print("=" * 70)

mismatches.to_csv(
    "nq_ohlcv_mismatches.csv",
    index=False
)

missing_old_ranges.to_csv(
    "nq_missing_from_old.csv",
    index=False
)

missing_new_ranges.to_csv(
    "nq_missing_from_new.csv",
    index=False
)

worst_price.head(1000).to_csv(
    "nq_worst_price_mismatches.csv",
    index=False
)

worst_volume.head(1000).to_csv(
    "nq_worst_volume_mismatches.csv",
    index=False
)

print("Reports saved.")

LOADING DATA

BASIC DATASET INFORMATION

NEW DATASET
Rows: 6540186
First: 2006-01-05 13:58:00
Last:  2026-08-07 20:59:00

OLD DATASET
Rows: 5411025
First: 2010-06-06 22:00:00
Last:  2026-06-01 23:59:00

DUPLICATE TIMESTAMPS
NEW duplicate rows: 0
OLD duplicate rows: 0

TIMESTAMP COMPARISON
Timestamps in NEW: 6540186
Timestamps in OLD: 5411025
Common timestamps: 5348930
Present in NEW but missing from OLD: 1191256
Present in OLD but missing from NEW: 62095

Missing from OLD - first 30 ranges:
                 start                 end  minutes
0  2006-01-05 13:58:00 2006-01-05 14:00:00        3
1  2006-01-05 14:02:00 2006-01-05 14:04:00        3
2  2006-01-05 14:06:00 2006-01-05 14:08:00        3
3  2006-01-05 14:10:00 2006-01-05 14:10:00        1
4  2006-01-05 14:13:00 2006-01-05 14:13:00        1
5  2006-01-05 14:15:00 2006-01-05 14:17:00        3
6  2006-01-05 14:19:00 2006-01-05 14:20:00        2
7  2006-01-05 14:23:00 2006-01-05 21:14:00      412
8  2006-01-06 10:23:00 2006-01-06 10

In [11]:
import pandas as pd
import numpy as np

# ================================================================
# FAST NQ OLD vs NEW DIAGNOSTIC
# ================================================================

OLD_FILE = "./nq_1min.parquet"
NEW_FILE = "./NQ_1min_output/continuous/NQ_continuous_raw.parquet"

START = pd.Timestamp("2020-10-16 20:59:00")

# Search for identical candles within +/- this many minutes.
SEARCH_MINUTES = 24 * 60


# ================================================================
# LOAD
# ================================================================

print("=" * 80)
print("LOADING")
print("=" * 80)

old = pd.read_parquet(OLD_FILE)
new = pd.read_parquet(NEW_FILE)

# OLD has datetime as index
if "datetime" not in old.columns:
    old = old.copy()
    old.index.name = "datetime"
    old = old.reset_index()

# NEW has datetime column
if "datetime" not in new.columns:
    if isinstance(new.index, pd.DatetimeIndex):
        new = new.copy()
        new.index.name = "datetime"
        new = new.reset_index()
    else:
        raise ValueError("NEW has no datetime column/index.")

old["datetime"] = pd.to_datetime(old["datetime"])
new["datetime"] = pd.to_datetime(new["datetime"])

old = old.sort_values("datetime").reset_index(drop=True)
new = new.sort_values("datetime").reset_index(drop=True)

old = old[old["datetime"] >= START].copy()
new = new[new["datetime"] >= START].copy()

print(f"OLD rows: {len(old):,}")
print(f"NEW rows: {len(new):,}")


# ================================================================
# MAKE EXACT OHLCV SIGNATURE
# ================================================================

cols = ["open", "high", "low", "close", "volume"]

for df in [old, new]:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

old["signature"] = list(
    zip(
        old["open"],
        old["high"],
        old["low"],
        old["close"],
        old["volume"],
    )
)

new["signature"] = list(
    zip(
        new["open"],
        new["high"],
        new["low"],
        new["close"],
        new["volume"],
    )
)


# ================================================================
# EXACT TIMESTAMP COMPARISON
# ================================================================

print()
print("=" * 80)
print("1. EXACT TIMESTAMP COMPARISON")
print("=" * 80)

old_i = old.set_index("datetime")
new_i = new.set_index("datetime")

common = old_i.index.intersection(new_i.index)

cmp = old_i.loc[
    common,
    cols
].join(
    new_i.loc[
        common,
        cols
    ],
    lsuffix="_old",
    rsuffix="_new"
)

cmp["open_equal"] = cmp["open_old"] == cmp["open_new"]
cmp["high_equal"] = cmp["high_old"] == cmp["high_new"]
cmp["low_equal"] = cmp["low_old"] == cmp["low_new"]
cmp["close_equal"] = cmp["close_old"] == cmp["close_new"]
cmp["volume_equal"] = cmp["volume_old"] == cmp["volume_new"]

cmp["ohlc_equal"] = (
    cmp["open_equal"]
    & cmp["high_equal"]
    & cmp["low_equal"]
    & cmp["close_equal"]
)

cmp["ohlcv_equal"] = (
    cmp["ohlc_equal"]
    & cmp["volume_equal"]
)

print(f"Common timestamps: {len(cmp):,}")
print(f"OHLC identical:    {cmp['ohlc_equal'].sum():,}")
print(f"OHLCV identical:   {cmp['ohlcv_equal'].sum():,}")
print(f"OHLC different:    {(~cmp['ohlc_equal']).sum():,}")
print(f"Volume different:  {(~cmp['volume_equal']).sum():,}")


# ================================================================
# CLASSIFY EACH COMMON TIMESTAMP
# ================================================================

cmp["category"] = "DIFFERENT_OHLC_AND_VOLUME"

cmp.loc[
    cmp["ohlc_equal"] & cmp["volume_equal"],
    "category"
] = "EXACT_OHLCV"

cmp.loc[
    cmp["ohlc_equal"] & ~cmp["volume_equal"],
    "category"
] = "OHLC_MATCH_VOLUME_DIFFERS"

cmp.loc[
    ~cmp["ohlc_equal"] & cmp["volume_equal"],
    "category"
] = "OHLC_DIFF_VOLUME_MATCH"


print()
print("CATEGORY COUNTS:")
print(cmp["category"].value_counts().to_string())


# ================================================================
# 2. IDENTIFY CONTIGUOUS PERIODS OF EXACT MATCHING
# ================================================================

print()
print("=" * 80)
print("2. MATCHING / DIVERGENCE PERIODS")
print("=" * 80)

x = cmp[["ohlc_equal", "ohlcv_equal"]].copy()

# Consecutive rows in cmp are not necessarily consecutive minutes.
# Therefore calculate timestamp gap first.

x["timestamp_gap"] = x.index.to_series().diff()

x["continuous_minute"] = (
    x["timestamp_gap"] == pd.Timedelta(minutes=1)
)

x["state"] = np.select(
    [
        x["ohlcv_equal"] & x["continuous_minute"],
        x["ohlc_equal"] & ~x["ohlcv_equal"],
        ~x["ohlc_equal"],
    ],
    [
        "EXACT_OHLCV",
        "OHLC_MATCH_VOLUME_DIFF",
        "OHLC_DIFFERENT",
    ],
    default="OTHER"
)

# Create groups when state changes OR timestamp is not 1 minute apart.
group_change = (
    x["state"].ne(x["state"].shift())
    | ~x["continuous_minute"]
)

x["group"] = group_change.cumsum()

periods = (
    x.groupby("group")
    .agg(
        start=("state", lambda s: s.index[0]),
        end=("state", lambda s: s.index[-1]),
        state=("state", "first"),
        candles=("state", "size"),
    )
    .reset_index(drop=True)
)

periods["duration"] = (
    periods["end"] - periods["start"]
)

print()
print("FIRST 30 PERIODS:")
display(periods.head(30))


# ================================================================
# 3. FOCUS ON YOUR FIRST DIVERGENCE
# ================================================================

FIRST_DIVERGENCE = pd.Timestamp("2020-10-18 22:00:00")

print()
print("=" * 80)
print("3. FIRST DIVERGENCE DETAIL")
print("=" * 80)

local_old = old[
    (old["datetime"] >= FIRST_DIVERGENCE - pd.Timedelta(minutes=10))
    &
    (old["datetime"] <= FIRST_DIVERGENCE + pd.Timedelta(minutes=30))
][["datetime"] + cols]

local_new = new[
    (new["datetime"] >= FIRST_DIVERGENCE - pd.Timedelta(minutes=10))
    &
    (new["datetime"] <= FIRST_DIVERGENCE + pd.Timedelta(minutes=30))
][["datetime"] + cols]

print("\nOLD:")
display(local_old)

print("\nNEW:")
display(local_new)


# ================================================================
# 4. FIND EXACT OHLCV CANDLE ELSEWHERE IN NEW
#
# Instead of expensive sequence matching, create a dictionary
# from signature -> timestamps.
# ================================================================

print()
print("=" * 80)
print("4. SEARCH FOR THE OLD DIVERGENCE CANDLES IN NEW")
print("=" * 80)

# Signature -> list of timestamps
new_lookup = {}

for row in new[["datetime", "signature"]].itertuples(index=False):

    new_lookup.setdefault(
        row.signature,
        []
    ).append(row.datetime)


# Inspect OLD candles around the divergence.

test_old = old[
    (old["datetime"] >= FIRST_DIVERGENCE - pd.Timedelta(minutes=5))
    &
    (old["datetime"] <= FIRST_DIVERGENCE + pd.Timedelta(hours=4))
].copy()

results = []

for row in test_old.itertuples(index=False):

    candidates = new_lookup.get(row.signature, [])

    for t in candidates:

        delta = t - row.datetime

        if abs(delta.total_seconds()) <= SEARCH_MINUTES * 60:

            results.append(
                {
                    "old_time": row.datetime,
                    "new_time": t,
                    "shift": delta,
                    "open": row.open,
                    "high": row.high,
                    "low": row.low,
                    "close": row.close,
                    "volume": row.volume,
                }
            )

shift_matches = pd.DataFrame(results)

if shift_matches.empty:

    print(
        "No identical OHLCV candles found within "
        f"+/- {SEARCH_MINUTES} minutes."
    )

else:

    print(
        f"Found {len(shift_matches):,} "
        "candidate identical OHLCV candles."
    )

    print()
    print("MOST COMMON SHIFTS:")

    print(
        shift_matches["shift"]
        .value_counts()
        .head(30)
        .to_string()
    )

    print()
    print("EXAMPLES:")

    display(
        shift_matches.head(100)
    )


# ================================================================
# 5. TEST SPECIFIC SHIFTS DIRECTLY
#
# This tells us whether NEW becomes identical to OLD after applying
# a fixed timestamp correction.
# ================================================================

print()
print("=" * 80)
print("5. FIXED-SHIFT TEST")
print("=" * 80)

# Test common possible shifts from -24h through +24h.
shift_results = []

old_lookup = old.set_index("datetime")[cols]

for minutes in range(
    -SEARCH_MINUTES,
    SEARCH_MINUTES + 1
):

    shifted_new = new.copy()

    shifted_new["shifted_datetime"] = (
        shifted_new["datetime"]
        + pd.Timedelta(minutes=minutes)
    )

    shifted_new = shifted_new.set_index(
        "shifted_datetime"
    )

    common_shifted = old_lookup.index.intersection(
        shifted_new.index
    )

    if len(common_shifted) == 0:
        continue

    a = old_lookup.loc[
        common_shifted
    ]

    b = shifted_new.loc[
        common_shifted,
        cols
    ]

    equal = (
        a["open"].to_numpy() == b["open"].to_numpy()
    ) & (
        a["high"].to_numpy() == b["high"].to_numpy()
    ) & (
        a["low"].to_numpy() == b["low"].to_numpy()
    ) & (
        a["close"].to_numpy() == b["close"].to_numpy()
    ) & (
        a["volume"].to_numpy() == b["volume"].to_numpy()
    )

    shift_results.append(
        {
            "shift_minutes": minutes,
            "common_rows": len(common_shifted),
            "exact_ohlcv": int(equal.sum()),
            "match_pct": (
                100 * equal.mean()
                if len(equal)
                else 0
            ),
        }
    )

shift_results = pd.DataFrame(shift_results)

print()
print(
    "BEST FIXED TIME SHIFTS:"
)

display(
    shift_results.sort_values(
        ["match_pct", "exact_ohlcv"],
        ascending=False
    ).head(30)
)


# ================================================================
# 6. FIND WHEN DATA RETURNS TO STRONG MATCHING
# ================================================================

print()
print("=" * 80)
print("6. MATCH RATE BY DAY")
print("=" * 80)

daily = cmp.copy()

daily["date"] = daily.index.date

daily_stats = (
    daily.groupby("date")
    .agg(
        candles=("ohlcv_equal", "size"),
        ohlcv_matches=("ohlcv_equal", "sum"),
        ohlc_matches=("ohlc_equal", "sum"),
    )
)

daily_stats["ohlcv_pct"] = (
    daily_stats["ohlcv_matches"]
    / daily_stats["candles"]
    * 100
)

daily_stats["ohlc_pct"] = (
    daily_stats["ohlc_matches"]
    / daily_stats["candles"]
    * 100
)

display(
    daily_stats[
        [
            "candles",
            "ohlcv_matches",
            "ohlcv_pct",
            "ohlc_matches",
            "ohlc_pct",
        ]
    ].head(30)
)

print()
print("WORST DAYS BY OHLCV MATCH RATE:")
display(
    daily_stats.sort_values(
        "ohlcv_pct"
    ).head(30)
)

print()
print("BEST DAYS BY OHLCV MATCH RATE:")
display(
    daily_stats.sort_values(
        "ohlcv_pct",
        ascending=False
    ).head(30)
)


# ================================================================
# 7. SAVE DIAGNOSTIC REPORTS
# ================================================================

cmp.reset_index().to_csv(
    "./NQ_old_vs_new_exact_timestamp_comparison.csv",
    index=False
)

periods.to_csv(
    "./NQ_old_vs_new_match_periods.csv",
    index=False
)

daily_stats.to_csv(
    "./NQ_old_vs_new_daily_match_rates.csv"
)

if not shift_matches.empty:
    shift_matches.to_csv(
        "./NQ_old_vs_new_shift_matches.csv",
        index=False
    )

shift_results.to_csv(
    "./NQ_old_vs_new_fixed_shift_test.csv",
    index=False
)


# ================================================================
# FINAL SUMMARY
# ================================================================

print()
print("=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

print(f"Starting point: {START}")

print(
    f"Exact OHLCV agreement: "
    f"{cmp['ohlcv_equal'].mean() * 100:.3f}%"
)

print(
    f"Exact OHLC agreement: "
    f"{cmp['ohlc_equal'].mean() * 100:.3f}%"
)

print(
    f"OHLCV disagreement: "
    f"{(~cmp['ohlcv_equal']).sum():,}"
)

print(
    f"OHLC disagreement: "
    f"{(~cmp['ohlc_equal']).sum():,}"
)

print()
print("Reports written:")
print("  NQ_old_vs_new_exact_timestamp_comparison.csv")
print("  NQ_old_vs_new_match_periods.csv")
print("  NQ_old_vs_new_daily_match_rates.csv")
print("  NQ_old_vs_new_shift_matches.csv")
print("  NQ_old_vs_new_fixed_shift_test.csv")

print()
print("=" * 80)
print("DONE")
print("=" * 80)

LOADING
OLD rows: 1,986,790
NEW rows: 2,050,452

1. EXACT TIMESTAMP COMPARISON
Common timestamps: 1,980,175
OHLC identical:    1,886,614
OHLCV identical:   1,870,380
OHLC different:    93,561
Volume different:  108,908

CATEGORY COUNTS:
category
EXACT_OHLCV                  1870380
DIFFERENT_OHLC_AND_VOLUME      92674
OHLC_MATCH_VOLUME_DIFFERS      16234
OHLC_DIFF_VOLUME_MATCH           887

2. MATCHING / DIVERGENCE PERIODS

FIRST 30 PERIODS:


,start,end,state,candles,duration
0,2020-10-16 20:59:00,2020-10-16 20:59:00,OTHER,1,0 days 00:00:00
1,2020-10-18 22:00:00,2020-10-19 09:03:00,OHLC_DIFFERENT,664,0 days 11:03:00
2,2020-10-19 09:18:00,2020-10-19 09:48:00,OHLC_DIFFERENT,31,0 days 00:30:00
3,2020-10-19 10:48:00,2020-10-19 20:14:00,OHLC_DIFFERENT,567,0 days 09:26:00
4,2020-10-19 20:30:00,2020-10-19 20:59:00,OHLC_DIFFERENT,30,0 days 00:29:00
5,2020-10-19 22:00:00,2020-10-20 09:03:00,OHLC_DIFFERENT,664,0 days 11:03:00
6,2020-10-20 09:18:00,2020-10-20 09:48:00,OHLC_DIFFERENT,31,0 days 00:30:00
7,2020-10-20 10:48:00,2020-10-20 20:14:00,OHLC_DIFFERENT,567,0 days 09:26:00
8,2020-10-20 20:30:00,2020-10-20 20:59:00,OHLC_DIFFERENT,30,0 days 00:29:00
9,2020-10-20 22:00:00,2020-10-21 09:03:00,OHLC_DIFFERENT,664,0 days 11:03:00



3. FIRST DIVERGENCE DETAIL

OLD:


,datetime,open,high,low,close,volume
3424236,2020-10-18 22:00:00,11816.75,11850.25,11816.75,11835.25,759
3424237,2020-10-18 22:01:00,11836.75,11840.50,11831.50,11831.75,255
3424238,2020-10-18 22:02:00,11832.75,11839.75,11830.25,11839.25,179
3424239,2020-10-18 22:03:00,11838.50,11851.25,11836.50,11849.50,381
3424240,2020-10-18 22:04:00,11849.00,11850.25,11845.75,11847.00,272
3424241,2020-10-18 22:05:00,11847.25,11858.50,11846.00,11853.25,487
3424242,2020-10-18 22:06:00,11853.75,11862.00,11853.25,11860.00,318
3424243,2020-10-18 22:07:00,11859.00,11867.00,11857.00,11862.00,281
3424244,2020-10-18 22:08:00,11861.50,11871.00,11861.00,11869.25,258
3424245,2020-10-18 22:09:00,11870.00,11872.50,11865.50,11866.25,207



NEW:


,datetime,open,high,low,close,volume
4489919,2020-10-18 21:50:00,11929.00,11930.25,11928.00,11929.50,27
4489920,2020-10-18 21:51:00,11929.25,11929.75,11927.00,11928.75,69
4489921,2020-10-18 21:52:00,11928.75,11931.00,11928.75,11928.75,38
4489922,2020-10-18 21:53:00,11929.50,11932.25,11928.75,11930.00,54
4489923,2020-10-18 21:54:00,11930.25,11932.25,11929.00,11931.25,46
4489924,2020-10-18 21:55:00,11931.50,11931.75,11929.75,11931.25,26
4489925,2020-10-18 21:56:00,11931.75,11937.50,11931.00,11934.00,148
4489926,2020-10-18 21:57:00,11934.50,11936.25,11933.50,11935.25,40
4489927,2020-10-18 21:58:00,11936.00,11936.25,11933.50,11935.25,27
4489928,2020-10-18 21:59:00,11935.75,11935.75,11931.75,11933.25,55



4. SEARCH FOR THE OLD DIVERGENCE CANDLES IN NEW
No identical OHLCV candles found within +/- 1440 minutes.

5. FIXED-SHIFT TEST


KeyboardInterrupt: 